# M7.2 · Continued Pre-Training (CPT) — production recipe, single GPU

This notebook runs the **same container-based recipe as the production CPT pipeline**
(`gsi-training/3.cpt/2.run_cpt/run-v2`), scaled to **one** A100 / H100 / H200.

**Same as production:**
- Engine: **NeMo AutoModel** inside the **`nvcr.io/nvidia/nemo-automodel:26.04`** container.
- Launch: `torchrun pretrain.py --config <recipe>.yaml` (the launcher is the
  verbatim 4-line production wrapper; everything is driven by the recipe YAML).
- Data: tokenized to a **Megatron-Core indexed dataset** (`.bin/.idx`) and read by
  the `MegatronPretraining` dataset — same path as production.

**Scaled for 1 GPU (the deltas):**
- Model: a small **dense Nemotron** (`nvidia/Nemotron-Mini-4B-Instruct`) instead of
  the 30B MoE — a 30B mixture-of-experts needs 8 GPUs; a 4B dense model full-finetunes
  on one 80 GB card.
- `distributed`: `dp_size: 1`, no expert-parallel (dense model), FSDP2 +
  activation checkpointing, BF16.
- Tiny `max_steps` and a small corpus so it finishes in minutes.

**Flow:** tokenize `data/cpt_corpus.jsonl` → `.bin/.idx` (in container) → write the
1-GPU recipe → `torchrun pretrain.py` (in container) → checkpoint in
`work/cpt_checkpoints/`, consumed by M7.3 SFT.


## 1. Prerequisites — GPU, NGC login, HF token, pull the container

No `pip` here: all training runs **inside** the NeMo AutoModel container, so the
only host requirements are Docker + a GPU. You need an **NGC API key** (to pull
the container) and an **HF token** (the Nemotron tokenizer/model is a gated repo).

In [1]:
import os, subprocess, getpass
from pathlib import Path

# --- GPU check (one recipe for any single A100 / H100 / H200) ---
import torch
assert torch.cuda.is_available(), "CPT needs a CUDA GPU (1x A100 / H100 / H200)."
_p = torch.cuda.get_device_properties(0)
print(f"GPU: {_p.name} ({_p.total_memory / 2**30:.1f} GiB)")
if _p.total_memory / 2**30 < 70:
    print("NOTE: full fine-tuning a 4B model wants ~48 GB+. On a <70 GB card, lower"
          " SEQ_LEN/steps or use a smaller model (set BASE_MODEL below).")

# --- Secrets (runtime prompts; not stored in the notebook) ---
if not os.environ.get("NGC_API_KEY"):
    os.environ["NGC_API_KEY"] = getpass.getpass("NGC API key: ").strip()
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("HuggingFace token (for the gated Nemotron repo): ").strip()
assert os.environ.get("NGC_API_KEY"), "NGC_API_KEY required to pull the container."

# --- Paths (this M7 folder is bind-mounted into the container at /workspace) ---
NB_DIR = Path.cwd().resolve()
CONTAINER = "nvcr.io/nvidia/nemo-automodel:26.04"
BASE_MODEL = "nvidia/Nemotron-Mini-4B-Instruct"   # small dense Nemotron; swap if you have less/more memory
(NB_DIR / "work").mkdir(exist_ok=True)

print("Docker login -> nvcr.io ...")
subprocess.run(["docker", "login", "nvcr.io", "-u", "$oauthtoken", "--password-stdin"],
               input=os.environ["NGC_API_KEY"].encode(), check=True)
print("Pulling", CONTAINER, "(first time: several GB) ...")
subprocess.check_call(["docker", "pull", CONTAINER])

# Reusable docker-run prefix: mount this folder at /workspace, 1 GPU, pass secrets.
DOCKER = [
    "docker", "run", "--rm", "--gpus", "device=0",
    "--shm-size=16g", "--ipc=host", "--ulimit", "memlock=-1",
    "-u", f"{os.getuid()}:{os.getgid()}", "--group-add", "0",
    "-e", "HOME=/tmp", "-e", "HF_HOME=/workspace/work/hf_cache",
    "-e", f"HF_TOKEN={os.environ['HF_TOKEN']}",
    "-e", "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True",
    "-v", f"{NB_DIR}:/workspace",
]
print("Container ready. Base model:", BASE_MODEL)


GPU: NVIDIA A100 80GB PCIe (79.3 GiB)


NGC API key:  ········
HuggingFace token (for the gated Nemotron repo):  ········


Docker login -> nvcr.io ...


WARNING! Your password will be stored unencrypted in /home/ubuntu/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credential-stores



Login Succeeded
Pulling nvcr.io/nvidia/nemo-automodel:26.04 (first time: several GB) ...
26.04: Pulling from nvidia/nemo-automodel
Digest: sha256:7213eab8055a2029ce1ef9022384a780f9095b9577f32edca4c48d303907421f
Status: Image is up to date for nvcr.io/nvidia/nemo-automodel:26.04
nvcr.io/nvidia/nemo-automodel:26.04
Container ready. Base model: nvidia/Nemotron-Mini-4B-Instruct


## 2. Stage 1 — tokenize the corpus to a Megatron indexed dataset

Production CPT trains on Megatron-Core `.bin/.idx` shards, not raw text. We run
`recipes/preprocess_cpt.py` **inside the container** (it uses NeMo AutoModel's
`indexed_dataset` builder and the model's tokenizer, with `--append-eod` marking
document boundaries) on `data/cpt_corpus.jsonl`. Output:
`work/cpt_data/native_processed_data_text_document.{bin,idx}`.

In [2]:
(NB_DIR / "work" / "cpt_data").mkdir(parents=True, exist_ok=True)
subprocess.check_call(DOCKER + [
    "--workdir", "/workspace/recipes", CONTAINER,
    "python3", "/workspace/recipes/preprocess_cpt.py",
    "--input", "/workspace/data/cpt_corpus.jsonl",
    "--output-prefix", "/workspace/work/cpt_data/native_processed_data",
    "--tokenizer", BASE_MODEL,
    "--json-key", "text",
])
print("tokenized shards:")
for f in sorted((NB_DIR / "work" / "cpt_data").glob("*")):
    print("  ", f.name, f"({f.stat().st_size} bytes)")



== PyTorch ==

NVIDIA Release 26.02 (build 305635159)
PyTorch Version 2.11.0a0+eb65b36
Container image Copyright (c) 2025, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
Copyright (c) 2014-2024 Facebook Inc.
Copyright (c) 2011-2014 Idiap Research Institute (Ronan Collobert)
Copyright (c) 2012-2014 Deepmind Technologies    (Koray Kavukcuoglu)
Copyright (c) 2011-2012 NEC Laboratories America (Koray Kavukcuoglu)
Copyright (c) 2011-2013 NYU                      (Clement Farabet)
Copyright (c) 2006-2010 NEC Laboratories America (Ronan Collobert, Leon Bottou, Iain Melvin, Jason Weston)
Copyright (c) 2006      Idiap Research Institute (Samy Bengio)
Copyright (c) 2001-2004 Idiap Research Institute (Ronan Collobert, Samy Bengio, Johnny Mariethoz)
Copyright (c) 2015      Google Inc.
Copyright (c) 2015      Yangqing Jia
Copyright (c) 2013-2016 The Caffe contributors
All rights reserved.

Various files include modifications (c) NVIDIA CORPORATION & AFFILIATES.  All rights reserved.

GOVERN

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


wrote 24 documents / 1325 tokens -> /workspace/work/cpt_data/native_processed_data_text_document.bin (+ .idx)
tokenized shards:
   native_processed_data_text_document.bin (5300 bytes)
   native_processed_data_text_document.idx (522 bytes)


## 3. Stage 2 — write the single-GPU CPT recipe

We generate the recipe YAML here so the `/workspace/...` paths resolve inside the
container. It mirrors the production `recipe_a100sxm-8.yaml` with the **1-GPU
deltas** called out in comments.

**Parameter significance (vs the production 8×A100 recipe):**
- `model.pretrained_model_name_or_path` — small **dense** Nemotron (was 30B MoE).
  We **omit** `use_mamba_kernels` (a dense `NemotronForCausalLM` rejects it; only the Mamba-hybrid Nemotron-H uses it).
- `dataset` — `MegatronPretraining` over our single `.bin/.idx` (production used a
  weighted **native/replay blend**; one tiny corpus needs no blend).
  `seq_length: 1024` (was 4096) to cut memory/time.
- `step_scheduler` — `global_batch_size: 8`, `local_batch_size: 1` (grad-accum 8),
  `max_steps: 40` (a mini-run; production = thousands).
- `distributed` — `dp_size: 1`, **no `ep_size`** (dense model, no experts),
  `activation_checkpointing: true` (memory), FSDP2, BF16.
- `optimizer` — AdamW `lr 1e-5`, `betas (0.9, 0.95)` (pretraining convention);
  cosine LR with short warmup — same shape as production.
- `fp8.enabled: false` — keep BF16 so the one recipe runs on A100 (no FP8 HW) too.

In [3]:
CPT_RECIPE = f'''# Single-GPU CPT recipe (workshop). Mirrors gsi-training/3.cpt/2.run_cpt/run-v2
# scaled to 1 GPU + a small dense Nemotron. Driven by recipes/pretrain.py.
model:
  _target_: nemo_automodel.NeMoAutoModelForCausalLM.from_pretrained
  pretrained_model_name_or_path: {BASE_MODEL}
  trust_remote_code: true
  torch_dtype: bfloat16
  # NOTE: no `use_mamba_kernels` -- that flag is only for the Mamba-hybrid
  # Nemotron-H (30B production model); a dense NemotronForCausalLM rejects it.
  # backend:
  #   _target_: nemo_automodel.components.models.common.BackendConfig
  #   linear: te
  #   rms_norm: torch_fp32
  #   enable_hf_state_dict_adapter: true
  #   enable_fsdp_optimizations: true

fp8:
  enabled: false                  # BF16 so the same recipe runs on A100 (no FP8 HW)

dataset:
  _target_: nemo_automodel.components.datasets.llm.megatron_dataset.MegatronPretraining
  paths:
    - /workspace/work/cpt_data/native_processed_data_text_document
  index_mapping_dir: /workspace/work/cpt_mapping
  tokenizer:
    _target_: transformers.AutoTokenizer.from_pretrained
    pretrained_model_name_or_path: {BASE_MODEL}
    trust_remote_code: true
  seq_length: 1024
  split: "0.9, 0.1, 0.0"
  splits_to_build: "train"

validation_dataset:
  _target_: nemo_automodel.components.datasets.llm.megatron_dataset.MegatronPretraining
  paths:
    - /workspace/work/cpt_data/native_processed_data_text_document
  index_mapping_dir: /workspace/work/cpt_mapping
  tokenizer:
    _target_: transformers.AutoTokenizer.from_pretrained
    pretrained_model_name_or_path: {BASE_MODEL}
    trust_remote_code: true
  seq_length: 1024
  split: "0.9, 0.1, 0.0"
  splits_to_build: "validation"
  num_val_samples: 32

step_scheduler:
  global_batch_size: 4            # effective batch (grad-accum = GBS / (LBS * dp))
  local_batch_size: 1
  ckpt_every_steps: 20
  val_every_steps: 20
  num_epochs: 1
  max_steps: 40                   # mini-run; production = thousands

dist_env:
  backend: nccl
  timeout_minutes: 30

rng:
  _target_: nemo_automodel.components.training.rng.StatefulRNG
  seed: 1111
  ranked: true

checkpoint:
  enabled: true
  checkpoint_dir: /workspace/work/cpt_checkpoints/
  model_save_format: safetensors
  save_consolidated: true

distributed:
  strategy: fsdp2
  dp_size: 1                      # single GPU (was dp_size: 8)
  dp_replicate_size: 1
  tp_size: 1
  cp_size: 1
  pp_size: 1
  activation_checkpointing: true  # memory: recompute activations in backward
  sequence_parallel: false

loss_fn:
  _target_: nemo_automodel.components.loss.masked_ce.MaskedCrossEntropy

dataloader:
  _target_: torchdata.stateful_dataloader.StatefulDataLoader
  collate_fn: torch.utils.data.default_collate
  num_workers: 0

validation_dataloader:
  _target_: torchdata.stateful_dataloader.StatefulDataLoader
  collate_fn: torch.utils.data.default_collate
  num_workers: 0

optimizer:
  _target_: torch.optim.AdamW
  betas: [0.9, 0.95]              # pretraining convention (lower beta2)
  lr: 1.0e-5                      # small LR: nudge, don't overwrite
  weight_decay: 0.1
  eps: 1.0e-8

lr_scheduler:
  lr_decay_style: cosine
  lr_warmup_steps: 5
  lr_decay_steps: 40
  min_lr: 1.0e-6
'''
(NB_DIR / "recipes" / "cpt_1gpu.yaml").write_text(CPT_RECIPE)
print("wrote recipes/cpt_1gpu.yaml")


wrote recipes/cpt_1gpu.yaml


## 4. Stage 3 — run CPT (`torchrun pretrain.py` in the container)

Same launch pattern as production, with `--nproc-per-node=1` for a single GPU.
This streams NeMo's training log; watch the per-step **loss** trend down and the
`lr` ramp (warmup) then decay (cosine). On the mini corpus this is minutes.

In [4]:
import sys
proc = subprocess.Popen(
    DOCKER + [
        "--workdir", "/workspace/recipes", CONTAINER,
        "torchrun", "--nproc-per-node=1",
        "/workspace/recipes/pretrain.py",
        "--config", "/workspace/recipes/cpt_1gpu.yaml",
    ],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:                       # stream the container log live
    sys.stdout.write(line)
rc = proc.wait()
print("\ntraining exited with code", rc)



== PyTorch ==

NVIDIA Release 26.02 (build 305635159)
PyTorch Version 2.11.0a0+eb65b36
Container image Copyright (c) 2025, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
Copyright (c) 2014-2024 Facebook Inc.
Copyright (c) 2011-2014 Idiap Research Institute (Ronan Collobert)
Copyright (c) 2012-2014 Deepmind Technologies    (Koray Kavukcuoglu)
Copyright (c) 2011-2012 NEC Laboratories America (Koray Kavukcuoglu)
Copyright (c) 2011-2013 NYU                      (Clement Farabet)
Copyright (c) 2006-2010 NEC Laboratories America (Ronan Collobert, Leon Bottou, Iain Melvin, Jason Weston)
Copyright (c) 2006      Idiap Research Institute (Samy Bengio)
Copyright (c) 2001-2004 Idiap Research Institute (Ronan Collobert, Samy Bengio, Johnny Mariethoz)
Copyright (c) 2015      Google Inc.
Copyright (c) 2015      Yangqing Jia
Copyright (c) 2013-2016 The Caffe contributors
All rights reserved.

Various files include modifications (c) NVIDIA CORPORATION & AFFILIATES.  All rights reserved.

GOVERN

## 5. Inspect the checkpoint

The consolidated checkpoint under `work/cpt_checkpoints/` is the CPT output that
M7.3 SFT fine-tunes.

In [5]:
import json
ckpt_root = NB_DIR / "work" / "cpt_checkpoints"
print("checkpoints:")
for d in sorted(ckpt_root.glob("*")):
    print("  ", d.name)
losses = sorted(ckpt_root.rglob("losses.json"))
if losses:
    data = json.loads(losses[-1].read_text())
    print("\nlosses.json:", str(data)[:400])
print("\nNext: M7.3 sft.ipynb fine-tunes this checkpoint.")


checkpoints:
   LATEST
   LOWEST_VAL
   epoch_0_step_19
   epoch_0_step_39
   training.jsonl
   validation.jsonl

losses.json: {'train_loss': 0.34967172145843506, 'val_loss': 0.3772149980068207}

Next: M7.3 sft.ipynb fine-tunes this checkpoint.
